In [1]:
import pandas as pd

In [2]:
df1=pd.read_csv("6000_train_examples.csv")
df2=pd.read_csv("extra_train_set.csv")

In [3]:
df=pd.concat([df1,df2],ignore_index=True)

In [4]:
df.head()

,prompt,A,B,C,D,E,answer
0,What is the primary role of Robin Juhkental in...,Robin Juhkental is the bassist of Malcolm Linc...,Robin Juhkental is the keyboardist of Malcolm ...,Robin Juhkental is the drummer of Malcolm Linc...,Robin Juhkental is the lead singer of Malcolm ...,Robin Juhkental is the lead guitarist of Malco...,D
1,Which of the following statements is true rega...,The theory of relativity only encompasses one ...,Special relativity explains the law of gravita...,The theory of relativity does not encompass an...,Special relativity applies to the cosmological...,General relativity only applies to the motion ...,D
2,In which country was the 1920 collection of co...,United States,Germany,Australia,France,England,E
3,What is one of the areas that Shimon Dovid Cow...,"Environmental conservation, opposing deforesta...","Homosexuality, looser abortion laws and volunt...","Freedom of speech, advocating for increased li...","Gun control, advocating for stricter regulatio...","Animal rights, opposing the use of animals for...",B
4,When did the Dirt Road Diaries Tour begin and ...,"February 17, 2014 - November 26, 2014","January 17, 2014 - October 26, 2014","February 17, 2013 - November 26, 2013","March 17, 2013 - December 26, 2013","January 17, 2013 - October 26, 2013",E


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   prompt  6500 non-null   object
 1   A       6499 non-null   object
 2   B       6499 non-null   object
 3   C       6500 non-null   object
 4   D       6500 non-null   object
 5   E       6498 non-null   object
 6   answer  6500 non-null   object
dtypes: object(7)
memory usage: 355.6+ KB


In [6]:
df.isna().sum()

prompt    0
A         1
B         1
C         0
D         0
E         2
answer    0
dtype: int64

In [7]:
df=df.dropna()

In [8]:
df.isna().sum()

prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [9]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
6495    False
6496    False
6497    False
6498    False
6499    False
Length: 6496, dtype: bool

In [10]:
def resolve_answer(row):
    if row["answer"] in df.columns:            #row["answer"]=the value of the column 'answer'  e.g., "E"
        return row[row["answer"]]              #row["answer"] is E ,so row["E"] is "England" so that returns "England"
    else:
        return row["answer"]                   #If answer is already text(England) so directly return England

df["answer"]=df.apply(resolve_answer,axis=1)   #Go row by row

                                               #Replace answer letter with actual text


In [11]:
df.head(2)

,prompt,A,B,C,D,E,answer
0,What is the primary role of Robin Juhkental in...,Robin Juhkental is the bassist of Malcolm Linc...,Robin Juhkental is the keyboardist of Malcolm ...,Robin Juhkental is the drummer of Malcolm Linc...,Robin Juhkental is the lead singer of Malcolm ...,Robin Juhkental is the lead guitarist of Malco...,Robin Juhkental is the lead singer of Malcolm ...
1,Which of the following statements is true rega...,The theory of relativity only encompasses one ...,Special relativity explains the law of gravita...,The theory of relativity does not encompass an...,Special relativity applies to the cosmological...,General relativity only applies to the motion ...,Special relativity applies to the cosmological...


In [12]:
df.drop(["A","B","C","D","E",],axis=1,inplace=True)

In [13]:
df.head(2)

,prompt,answer
0,What is the primary role of Robin Juhkental in...,Robin Juhkental is the lead singer of Malcolm ...
1,Which of the following statements is true rega...,Special relativity applies to the cosmological...


In [14]:
df.shape

(6496, 2)

In [15]:
df.isnull().sum()

prompt    0
answer    0
dtype: int64

In [16]:
df["text"]="Question : "+df["prompt"]+"\nAnswer : "+df["answer"]
df.drop(["prompt","answer"],axis=1,inplace=True)

In [17]:
df.head(3)

,text
0,Question : What is the primary role of Robin J...
1,Question : Which of the following statements i...
2,Question : In which country was the 1920 colle...


In [18]:
from sklearn.model_selection import train_test_split

train_df,val_df=train_test_split(df,test_size=0.2,random_state=42)

Load Tokenizer

In [19]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token=tokenizer.eos_token



Tokenize Dataset



In [20]:
def tokenize_function(examples):
    return tokenizer(          #a tokenizer object from the Hugging Face Transformers library.The tokenizer converts text → numbers (tokens) that the model can understand
        examples,              #This is the text input that will be tokenized.
        padding="max_length",  #Padding makes all sentences the same length.
        truncation=True,       # If the sentence is longer than the maximum length, it will be cut (truncated).
        max_length=128         
        
    )

In [21]:
train_encodings=tokenize_function(train_df["text"].tolist())
val_encodings=tokenize_function(val_df["text"].tolist())

#Tokenized training data  → train_encodings
#Tokenized validation data → val_encodings

Create PyTorch Dataset Class


In [22]:
import torch

class QADataset(torch.utils.data.Dataset):
  
    def __init__(self,encodings):    #dataset=QADataset(train_encodings), so that self=dataset
                                     #encoding is a variable name it contains tokenized data (train_encodings)
        self.encodings=encodings     #Store the encodings inside the dataset object

    def __getitem__(self,idx):       #idx is the row number of the dataset

        item={key: torch.tensor(val[idx]) for key, val in self.encodings.items()}  #DL models only works in tensor
        item["labels"]=item["input_ids"]
        return item
        

    def __len__(self):
        return len(self.encodings["input_ids"])

#QAdataset is the class and (torch.utils.data.Dataset) is a parent class
#is a built-in dataset class provided by PyTorch.
#Create a dataset class called QADataset. that inherits PyTorch's Dataset class



#"input_ids ":torch.tensor(self.encodings["input_ids"][idx]),  #DL models only works in tensor
#            "attention_mask":torch.tensor(self.encodings["attention_mask"][idx]),
#            "labels":torch.tensor(self.encodings["input_ids"][idx])

      LOAD MODEL


CausalLM = Generative model

In [23]:
from transformers import AutoModelForCausalLM
model=AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

    TRAINING SETUP

In [24]:
from transformers import TrainingArguments, Trainer

training_args=TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="./logs",
    save_strategy="epoch",
)


#Train for 3 epochs
#Use batch size 4
#Evaluate after each epoch
#Save model after each epoch
#Store outputs in results folder
#Store logs in logs folder

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=QADataset(train_encodings),
    eval_dataset=QADataset(val_encodings)
)
trainer.train()

C:\Users\ADMIN\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.862265,0.815142
2,0.732697,0.813873
3,0.658924,0.824529


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\ADMIN\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\ADMIN\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3897, training_loss=0.7561293815630413, metrics={'train_runtime': 36775.3737, 'train_samples_per_second': 0.424, 'train_steps_per_second': 0.106, 'total_flos': 1018255048704000.0, 'train_loss': 0.7561293815630413, 'epoch': 3.0})

In [31]:
prompt="Question: What is 2+2?\nAnswer:"
inputs=tokenizer(prompt,return_tensors="pt")
outputs=model.generate(
    **inputs,
    max_length=50
) 
print(tokenizer.decode(outputs[0],skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What is 2+2?
Answer: 2+2 is a term used to describe a measure of the relative stability of a solid object in a vacuum.


In [32]:
trainer.save_model("./trained_model")
tokenizer.save_pretrained("./trained_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./trained_model\\tokenizer_config.json', './trained_model\\tokenizer.json')

In [42]:
from transformers import pipeline

generator=pipeline(
    "text-generation",
    model="./trained_model",
    tokenizer=tokenizer
)

print(generator("what is artificial intelligence",max_length=50))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'what is artificial intelligence (AI)?\nAnswer : A field in which artificial intelligence (AI) is developed and implemented to improve human intelligence.'}]


In [47]:

trainer.evaluate()


RuntimeError: on_train_begin must be called before on_evaluate

In [1]:
from transformers import TrainerCallback

class DisableProgressCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, **kwargs):
        return control

trainer.remove_callback("NotebookProgressCallback")
trainer.add_callback(DisableProgressCallback())
metrics = trainer.evaluate()

NameError: name 'trainer' is not defined